<a href="https://colab.research.google.com/github/Mmbsaksd/Full_Stack_GenerativeAI_And_Agentic_AI/blob/main/vision_model_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U datasets huggingface_hub pillow

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached huggingface_hub-1.23.0-py3-none-any.whl.metadata (14 kB)
Using cached datasets-5.0.0-py3-none-any.whl (555 kB)
Using cached huggingface_hub-1.23.0-py3-none-any.whl (770 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
transformers 4.57.1 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.23.0 which 

In [2]:
import os
from datasets import Dataset, Features, Value, Image

IMG_DIR = "./iphone_imgs"
image_files = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])
captions = [
    "A white iPhone shown from the back and front on a white background.",
    "A dark blue iPhone shown from the back with a side view on a transparent background.",
    "An orange iPhone shown from the back and front on a white background.",
    "Three iPhones in white, orange, and dark blue shown together from the back.",
    "An orange iPhone shown from the back on a transparent background.",
]

In [3]:
image_files

['./iphone_imgs/image01.jpg',
 './iphone_imgs/image02.jpg',
 './iphone_imgs/image03.jpg',
 './iphone_imgs/image04.jpg',
 './iphone_imgs/image05.jpg']

In [4]:
instruction = "Describe this iPhone product image in one sentence."

In [5]:
rows = []

for img_path, cap in zip(image_files, captions):
    rows.append({
        "image": img_path,
        "text": cap,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": img_path},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ],
    })

In [6]:
rows

[{'image': './iphone_imgs/image01.jpg',
  'text': 'A white iPhone shown from the back and front on a white background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs/image01.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A white iPhone shown from the back and front on a white background.'}]}]},
 {'image': './iphone_imgs/image02.jpg',
  'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs/image02.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.'}]}]},
 {'image': './iphone_imgs/imag

In [7]:
features = Features({
    "image": Image(),
    "text": Value("string"),
    "messages": Value("string"),  # keep as JSON string OR store raw python objects separately
})

In [8]:
ds = Dataset.from_list(rows, features=features)

In [9]:
splits = ds.train_test_split(test_size=1, seed=3407)

In [10]:
from huggingface_hub import login
REPO_ID = "mmbsaksd/iphone5_vlm_img"

You need to log in to Hugging Face Hub to push datasets. Please run the following cell and enter your Hugging Face token when prompted. You can find your token in your Hugging Face settings under 'Access Tokens'.

In [11]:
#login()

In [12]:
splits["train"].push_to_hub(REPO_ID, split="train")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp6tl3oyru.parquet    : 100%|##########| 26.3kB / 26.3kB            

CommitInfo(commit_url='https://huggingface.co/datasets/mmbsaksd/iphone5_vlm_img/commit/95fd031b289999bdc1caf39493ae107c9ab71250', commit_message='Upload dataset', commit_description='', oid='95fd031b289999bdc1caf39493ae107c9ab71250', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mmbsaksd/iphone5_vlm_img', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mmbsaksd/iphone5_vlm_img'), pr_revision=None, pr_num=None)

In [13]:
splits["test"].push_to_hub(REPO_ID, split="test")

Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp2ku7rh27.parquet    : 100%|##########| 10.6kB / 10.6kB            

CommitInfo(commit_url='https://huggingface.co/datasets/mmbsaksd/iphone5_vlm_img/commit/9de28d2084b84c7a097831b7c5eaff3ec45f6e08', commit_message='Upload dataset', commit_description='', oid='9de28d2084b84c7a097831b7c5eaff3ec45f6e08', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/mmbsaksd/iphone5_vlm_img', endpoint='https://huggingface.co', repo_type='dataset', repo_id='mmbsaksd/iphone5_vlm_img'), pr_revision=None, pr_num=None)

In [14]:
print("Uploaded:", REPO_ID)

Uploaded: mmbsaksd/iphone5_vlm_img


In [15]:
from datasets import load_dataset

In [16]:
dataset = load_dataset("HuggingFaceM4/ChartQA")

In [17]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 28299
    })
    val: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 2500
    })
})


In [18]:
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")

In [19]:
print(train_ds)

Dataset({
    features: ['image', 'query', 'label', 'human_or_machine'],
    num_rows: 28299
})


In [20]:
print(train_ds[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359 at 0x7858A40E6D20>, 'query': 'Is the value of Favorable 38 in 2015?', 'label': ['Yes'], 'human_or_machine': 0}


In [21]:
# --- Install (Colab) ---
!pip -q install "transformers==4.57.1" --upgrade
!pip -q install --no-deps trl==0.22.2
!pip -q install unsloth unsloth_zoo bitsandbytes accelerate peft triton
!pip -q install sentencepiece protobuf datasets huggingface_hub hf_transfer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.7.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth 2026.7.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [22]:

# ==========================================================
# Universal Unsloth Vision Fine-tuning (Model + Dataset Dynamic)
# Supports: Qwen VL, LLaVA, Pixtral, MedGemma, Llama Vision
# Dataset selectable
# Trains on 150 row subset
# Runs for 2 Epochs
# ==========================================================

import unsloth
import os
import torch
from dataclasses import dataclass
from typing import Dict

from datasets import load_dataset
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [23]:
# ==========================================================
# 1️Model Registry
# ==========================================================
MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}

In [24]:
# ==========================================================
# Dataset Registry (Vision-ready datasets)
# ==========================================================
DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },

    "iphone_custom": {
    "name": "sunny199/iphone5_vlm",  # change to your HF repo
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

In [25]:
# ==========================================================
# Config
# ==========================================================
@dataclass
class VisionFTConfig:
    model_key: str = "qwen2_vl_2b"
    dataset_key: str = "iphone_custom"

    subset_rows: int = 150
    eval_ratio: float = 0.1 #10% data for evaluation
    seed: int = 3407

    # LoRA
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    weight_decay: float = 0.001
    max_length: int = 2048

    output_dir: str = "outputs"
    save_dir: str = "vlm_lora_output"

In [26]:
# ==========================================================
# Trainer Class
# ==========================================================
class VisionFineTuner:

    def __init__(self, cfg: VisionFTConfig):
        self.cfg = cfg

        # Get model and dataset details from registry
        self.model_name = MODEL_REGISTRY[cfg.model_key]
        self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

        self.model = None
        self.tokenizer = None
        self.train_ds = None
        self.eval_ds = None
        self.trainer = None

    # -----------------------------
    # Load Model + Apply LoRA
    # -----------------------------
    def load_model(self):
        print("Loading model:", self.model_name)

        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            self.model_name,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth",
        )

        self.model = FastVisionModel.get_peft_model(
            self.model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=self.cfg.r,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            random_state=self.cfg.seed,
        )

        print("Model loaded and LoRA applied.")
        return self

    # -----------------------------
    # Prepare Dataset
    # -----------------------------
    def prepare_data(self):
        print("Loading dataset:", self.dataset_info["name"])

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        # Use small subset for demo/testing
        raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))

        instruction = self.dataset_info["instruction"]
        image_key = self.dataset_info["image_key"]
        text_key = self.dataset_info["text_key"]

        def format_sample(example):
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

        ds = raw.map(
            format_sample,
            remove_columns=raw.column_names
        )

        splits = ds.train_test_split(
            test_size=self.cfg.eval_ratio,
            seed=self.cfg.seed
        )

        self.train_ds = splits["train"]
        self.eval_ds = splits["test"]

        print("Train samples:", len(self.train_ds))
        print("Eval samples:", len(self.eval_ds))

        return self

    # -----------------------------
    # Build Trainer
    # -----------------------------
    def build_trainer(self):
        print("Building trainer...")

        FastVisionModel.for_training(self.model)

        training_args = SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
            num_train_epochs=self.cfg.num_train_epochs,
            learning_rate=self.cfg.learning_rate,
            logging_steps=self.cfg.logging_steps,
            optim="adamw_8bit",
            weight_decay=self.cfg.weight_decay,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none",

            # Important for vision-language fine-tuning
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},

            max_length=self.cfg.max_length,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=UnslothVisionDataCollator(
                self.model,
                self.tokenizer
            ),
            train_dataset=self.train_ds,
            eval_dataset=self.eval_ds,
            args=training_args,
        )

        print("Trainer ready.")
        return self

    # -----------------------------
    # Train Model
    # -----------------------------
    def train(self):
        print("Training started for", self.cfg.num_train_epochs, "epochs")
        self.trainer.train()
        print("Training completed.")
        return self

    # -----------------------------
    # Save Model
    # -----------------------------
    def save(self):
        os.makedirs(self.cfg.save_dir, exist_ok=True)

        self.model.save_pretrained(self.cfg.save_dir)
        self.tokenizer.save_pretrained(self.cfg.save_dir)

        print("Model saved to:", self.cfg.save_dir)
        return self

    # -----------------------------
    # Quick Inference Test
    # -----------------------------
    def quick_infer(self, sample_index=0):
        print("Running quick inference...")

        FastVisionModel.for_inference(self.model)

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        image = raw[sample_index][self.dataset_info["image_key"]]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": self.dataset_info["instruction"]},
                    {"type": "image"},
                ],
            }
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = self.tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextStreamer(
            self.tokenizer,
            skip_prompt=True
        )

        self.model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=128,
            temperature=1.2,
            do_sample=True,
        )

        return self

    # -----------------------------
    # Full Pipeline Runner
    # -----------------------------
    def run(self):
        self.load_model()
        self.prepare_data()
        self.build_trainer()
        self.train()
        self.save()
        self.quick_infer()

In [27]:
cfg = VisionFTConfig(
    model_key="qwen2_vl_2b",
    dataset_key="iphone_custom",
)

In [28]:
trainer = VisionFineTuner(cfg)

In [29]:
trainer.load_model()

Loading model: unsloth/Qwen2-VL-2B-Instruct-bnb-4bit
==((====))==  Unsloth 2026.7.2: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
Unsloth: Warning - VLM processor fallback returned None for model_type=qwen2_vl


RuntimeError: Unsloth: Could not load the tokenizer/processor. If you are offline, make sure the tokenizer files exist in the checkpoint folder or were previously downloaded to the Hugging Face cache, or set HF_HUB_OFFLINE=1 to force local loading. Otherwise please check that the model has a tokenizer.

In [ ]:
trainer.prepare_data()

In [ ]:
trainer.build_trainer()

In [ ]:
trainer.train()